## Results of running the LLM as Judge on the syntheitc datasets

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path



: 

## Individual task correctness scores
This chart displays each task's correctness score for all of the synthetic dataset across the 4 dimensions the Judge scored on

In [ ]:
RESULTS_DIR = Path("llm_audit_results")

tasks = pd.read_csv(RESULTS_DIR / "task_scores.csv")
domains = pd.read_csv(RESULTS_DIR / "domain_scores.csv")

dimensions = [
    "task_consistency",
    "expected_outcome_validity",
    "policy_compliance",
    "workflow_feasibility"
]

display(tasks.head())

# Average correctness by dimension
This chart shows which of the four dimensions contributes most to the dataset's correctness score.

In [ ]:

dataset = "banking_customer_onboarding"

dataset_tasks = tasks[tasks["domain"] == dataset].copy()

plt.figure(figsize=(10, 5))

plt.bar(
    dataset_tasks["task_id"],
    dataset_tasks["task_score"] * 100
)

plt.axhline(
    dataset_tasks["task_score"].mean() * 100,
    color="red",
    linestyle="--",
    label=f"Average: {dataset_tasks['task_score'].mean():.1%}"
)

plt.title(f"Task Correctness — {dataset}")
plt.xlabel("Task ID")
plt.ylabel("Correctness Score (%)")
plt.ylim(0, 100)
plt.xticks(rotation=35, ha="right")
plt.legend()
plt.tight_layout()
plt.show()


# Average correctness across all synthetic datasets
This chart compares the LLM as a judge correctness score across all of the datasets. 

In [ ]:

dimension_scores = dataset_tasks[dimensions].mean() * 100

plt.figure(figsize=(9, 5))

plt.bar(
    [d.replace("_", " ").title() for d in dimensions],
    dimension_scores
)

plt.title(f"Correctness by Dimension — {dataset}")
plt.ylabel("Average Score (%)")
plt.ylim(0, 100)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


# Average correctness across all synthetic datasets
This chart compares the dataset-level scores.

In [ ]:

dataset_scores = (
    domains[domains["domain"] != "OVERALL"]
    .sort_values("mean_correctness")
)

plt.figure(figsize=(10, 5))

plt.barh(
    dataset_scores["domain"],
    dataset_scores["mean_correctness"] * 100
)

plt.title("LLM Correctness Across Synthetic Datasets")
plt.xlabel("Average Correctness Score (%)")
plt.xlim(0, 100)
plt.tight_layout()
plt.show()


# Summary table
Displays the number of tasks evaluated and the average correctness score for each dataset.

In [ ]:

summary = domains.copy()

summary["mean_correctness"] = (
    summary["mean_correctness"] * 100
).round(2)

summary = summary.rename(columns={
    "domain": "Dataset",
    "task_count": "Tasks Evaluated",
    "mean_correctness": "Average Correctness (%)",
    "uncertain_dimensions": "Uncertain Judgments",
    "violated_dimensions": "Incorrect Judgments"
})

display(summary)


# Visualize all four dimensions for each task
This grouped bar chart is useful for identifying exactly which correctness dimensions are lowering individual task scores.

In [ ]:

plot_data = dataset_tasks.set_index("task_id")[dimensions] * 100

ax = plot_data.plot(
    kind="bar",
    figsize=(12, 6),
    width=0.8
)

plt.title(f"Dimension Scores by Task — {dataset}")
plt.xlabel("Task ID")
plt.ylabel("Correctness Score (%)")
plt.ylim(0, 110)
plt.xticks(rotation=35, ha="right")
plt.legend(
    title="Correctness Dimension",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)
plt.tight_layout()
plt.show()
